In this notebook, we extend the agent state through various middlewares. The goal is to enable short-term memory within a single invocation.

In [ ]:
!pip install -q langchain langchain-openai

In [ ]:
import operator

from google.colab import userdata
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import after_model
from langchain.agents.middleware import TodoListMiddleware
from langchain.messages import AIMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.runtime import Runtime
from typing import Annotated, List


api_key = userdata.get('OPENAI_API_KEY')


def print_conversation(conversation: List[AIMessage]):
    for message in conversation:
        message.pretty_print()

In [ ]:
class TrackUsageState(AgentState):
    input_tokens: Annotated[int, operator.add]
    cached_tokens: Annotated[int, operator.add]
    output_tokens: Annotated[int, operator.add]
    reasoning_tokens: Annotated[int, operator.add]


@after_model(state_schema=TrackUsageState)
def track_usage(state: TrackUsageState, runtime: Runtime):
    last_message = state["messages"][-1]
    if last_message.usage_metadata is None:
        return None

    state_update = { "input_tokens": last_message.usage_metadata["input_tokens"], "output_tokens": last_message.usage_metadata["output_tokens"] }

    if "input_token_details" in last_message.usage_metadata:
        state_update["cached_tokens"] = last_message.usage_metadata["input_token_details"]["cache_read"]
    if "output_token_details" in last_message.usage_metadata is not None:
        state_update["reasoning_tokens"] = last_message.usage_metadata["output_token_details"]["reasoning"]

    return state_update

In [ ]:
class CountModelCallsState(AgentState):
    model_calls: Annotated[int, operator.add]


@after_model(state_schema=CountModelCallsState)
def count_model_calls(state: CountModelCallsState, runtime: Runtime):
    return { "model_calls": 1 }

In [ ]:
agent = create_agent(
    model=ChatOpenAI(model="gpt-5-nano", api_key=api_key, reasoning_effort="low"),
    middleware=[track_usage, count_model_calls, TodoListMiddleware()]
)

In [ ]:
prove_irrational = agent.invoke(
    input={
        "messages": [
            HumanMessage("Prove that the square roots of 2 and 3 are irrational. First plan your actions, prepare a list of TODO items and then start working them one by one. I want you to provide at least 2 independent proofs for both tasks.")
        ]
    }
)

In [ ]:
print_conversation(prove_irrational['messages'])

In [ ]:
print(f"Model calls: {prove_irrational['model_calls']}")
print(f"Input tokens: {prove_irrational['input_tokens']}; Cache read: {prove_irrational['cached_tokens']}")
print(f"Output tokens: {prove_irrational['output_tokens']}; Reasoning: {prove_irrational['reasoning_tokens']}")